# Vibepixel Avatar Pipeline

Este notebook sigue el flujo de Colab para importar la planilla, previsualizar el avatar y exportar. La primera celda intenta usar `avatar_pipeline.py` local y, si falta, lo descarga automáticamente desde el repositorio público con un tiempo de espera corto. La celda del CSV también baja un ejemplo en la primera corrida si no encuentra `avatar-16x16.csv`, así podés ver un resultado sin pasos extra. Mantené los valores RGB entre comillas como un solo campo CSV, por ejemplo `"255,204,0"`.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import urllib.request

AVATAR_PIPELINE_URL = "https://raw.githubusercontent.com/HaroldSthid/VibePixel-AcademyBabyStepsTech/main/artifacts/colab/avatar_pipeline.py"
SAMPLE_CSV_URL = "https://raw.githubusercontent.com/HaroldSthid/VibePixel-AcademyBabyStepsTech/main/artifacts/spreadsheet/templates/avatar-16x16.csv"
MODULE_PATH = Path("avatar_pipeline.py")
DOWNLOAD_TIMEOUT_SECONDS = 10
DOWNLOAD_ATTEMPTS = 2

def _download_file(url, destination, *, label):
    last_exc = None
    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        try:
            with urllib.request.urlopen(url, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
                destination.write_bytes(response.read())
            return
        except Exception as download_exc:
            last_exc = download_exc
            if attempt < DOWNLOAD_ATTEMPTS:
                print(f"No se pudo descargar {label} en el intento {attempt}. Reintentamos una vez más.")
    raise RuntimeError(
        f"No pudimos descargar {label} automáticamente en {DOWNLOAD_TIMEOUT_SECONDS} segundos por intento. "
        "Revisá tu conexión a internet y volvé a ejecutar la celda."
    ) from last_exc

def _load_avatar_pipeline():
    try:
        from avatar_pipeline import export_avatar, load_avatar_frames, preview_avatar
        return export_avatar, load_avatar_frames, preview_avatar
    except ImportError:
        if not MODULE_PATH.exists():
            try:
                _download_file(AVATAR_PIPELINE_URL, MODULE_PATH, label="avatar_pipeline.py")
            except RuntimeError as download_exc:
                raise ImportError(
                    "No pudimos cargar avatar_pipeline.py automáticamente. Verificá tu conexión a internet y volvé a ejecutar la celda."
                ) from download_exc

        try:
            spec = importlib.util.spec_from_file_location("avatar_pipeline", MODULE_PATH)
            if spec is None or spec.loader is None:
                raise ImportError("No se pudo preparar la carga de avatar_pipeline.py.")
            module = importlib.util.module_from_spec(spec)
            sys.modules["avatar_pipeline"] = module
            spec.loader.exec_module(module)
            return module.export_avatar, module.load_avatar_frames, module.preview_avatar
        except Exception as load_exc:
            raise ImportError(
                "No pudimos cargar avatar_pipeline.py automáticamente. Verificá que el notebook tenga acceso a internet y volvé a ejecutar la celda."
            ) from load_exc

export_avatar, load_avatar_frames, preview_avatar = _load_avatar_pipeline()


In [ ]:
source_csv = Path("avatar-16x16.csv")  # Reemplazalo después por tu propia exportación si querés.
if not source_csv.exists():
    try:
        _download_file(SAMPLE_CSV_URL, source_csv, label="avatar-16x16.csv")
        print("No encontramos avatar-16x16.csv, así que cargamos un ejemplo para que puedas seguir. Después podés reemplazarlo por tu propio CSV exportado.")
    except RuntimeError as download_exc:
        raise FileNotFoundError("No pudimos descargar el CSV de ejemplo automáticamente. Revisá tu conexión, subí tu propio CSV exportado y volvé a ejecutar esta celda.") from download_exc
frames = load_avatar_frames(source_csv)
len(frames), [frame_id for frame_id, _ in frames]


In [ ]:
preview = preview_avatar(frames, scale=16)
preview


In [ ]:
result = export_avatar(frames, output_dir=Path("exports"), stem="avatar")
result


## Fallback para una sola imagen

Si la exportación contiene un solo frame, el pipeline escribe `avatar_preview.png` en lugar de un GIF animado. Usá esa imagen fija para el siguiente paso del taller. Ejecutá `validate_avatar_pipeline.py` para verificar el contrato de importación, previsualización y exportación antes de compartir el artefacto. La celda de importación ya maneja automáticamente el fallback de descarga.
